In [ ]:
REPO_COMMIT = 'c2499f306c41d143db75e8135ab79d7f3f51e325'
SOURCE_DIGEST = None
RUN_TAG = 'vna_v5_inc10_p2'
ARMS = ('icarl_full', 'icarl', 'icarl_cbp', 'icarl_redo')
JOBS_GPU0 = ['icarl_full__nobn__superclass__inc10__p2__s0']
JOBS_GPU1 = ['icarl__nobn__superclass__inc10__p2__s0','icarl_cbp__nobn__superclass__inc10__p2__s0','icarl_redo__nobn__superclass__inc10__p2__s0']
import time
SESSION_HOURS = 11.5
SESSION_START = time.time()
VNA_DEADLINE = SESSION_START + SESSION_HOURS*3600
DISK_CAP_BYTES = 20_000_000_000


# VNA v5 — B0 Inc10 / P2 / seed-0 CertSwap pilot
Enable Internet, T4×2 and the `github_token` secret. Attach CIFAR-100 with `cifar-100-python/{train,test,meta}` and this pilot's own prior output archive to resume. When SOURCE_DIGEST is set, attach the exact source ZIP matching that digest; the base Git pin alone does not contain the local changes.

All four arms use ResNet32 noBN/SkipInit, superclass order, memory 2000, native 100/50 epochs, lr scale 0.5 and one epoch of linear warmup at EVERY task; clipping stays off. GPU0 runs a fresh Inc10/P2 CertSwap authority preflight and then full seed 0 (Gram/power10). GPU1 starts baseline seed 0; available GPUs then draw CBP/ReDo from the shared queue, using their registered original defaults. No seeds 1–2 or other competitors are launched.

Task-2 and task-10 reports appear as soon as each arm reaches them and do not stop training. The report retains CertSwap acceptance/rejection counts, separate VNA/CertSwap timing and per-site dead/dormant fractions on the native fixed 256-sample probe. Output: `outputs/vna_v5_inc10_p2/`. P1, clipping studies and all old notebook outputs are kept separate.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
os.environ.update(TQDM_DISABLE="1", PYTHONUNBUFFERED="1")
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
RUN_ROOT = Path("/kaggle/working/vna5_inc10_p2_pilot")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if len(branch) == 40 and all(c in "0123456789abcdef" for c in branch):
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", branch], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", branch], env=env, check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO,):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)

if SOURCE_DIGEST:
    bundles = [p for p in Path('/kaggle/input').rglob('vna_v3_source.zip')
               if hashlib.sha256(p.read_bytes()).hexdigest() == SOURCE_DIGEST]
    if not bundles: raise FileNotFoundError('Attach vna_v3_source.zip matching SOURCE_DIGEST; this notebook uses uncommitted source')
    with zipfile.ZipFile(bundles[0]) as archive:
        for name in archive.namelist():
            target = (REPO/name).resolve()
            if not target.is_relative_to(REPO.resolve()): raise ValueError('unsafe source bundle member')
        archive.extractall(REPO)
    os.environ['VNA_SOURCE_DIGEST'] = SOURCE_DIGEST
    shutil.copy2(bundles[0], RUN_ROOT/'vna_v3_source.zip')
    (RUN_ROOT/'source_manifest.json').write_text(json.dumps(dict(code_commit=REPO_COMMIT, source_digest=SOURCE_DIGEST)))


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONUNBUFFERED="1", TQDM_DISABLE="1", PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
import torch
if torch.cuda.device_count() != 2:
    raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
required_cifar_files = {"train", "test", "meta"}
cifar_dirs = []
for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
    path = Path(directory)
    if (path.name == "cifar-100-python" and
            required_cifar_files.issubset(filenames)):
        cifar_dirs.append(path.parent)
cifar_dirs = sorted(set(cifar_dirs), key=str)
if not cifar_dirs:
    visible = []
    for directory, dirnames, filenames in os.walk(
            "/kaggle/input", followlinks=True):
        visible.append({
            "directory": directory,
            "directories": sorted(dirnames)[:10],
            "files": sorted(filenames)[:10],
        })
        if len(visible) == 20:
            break
    raise FileNotFoundError(
        "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
        f"Visible input inventory: {visible}")
DATA_ROOT = cifar_dirs[0]
print("CIFAR-100 root:", DATA_ROOT)

if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO)]
print("Official TEST verified:", DATA_ROOT / "cifar-100-python/test")



In [ ]:
from concurrent.futures import ThreadPoolExecutor
from experiments.vna_kaggle import restore_stage_archives
from experiments.vna_kaggle import archive_outputs as _archive_outputs
STAGE_B = RUN_ROOT / "outputs"
STAGE_B.mkdir(parents=True, exist_ok=True)
ROOT=STAGE_B/RUN_TAG
ROOT.mkdir(parents=True,exist_ok=True)
restore_stage_archives(Path('/kaggle/input'), ROOT, RUN_TAG)
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
       "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python", "VNA_DEADLINE": str(VNA_DEADLINE)}
session_paused = threading.Event()
lane_state = {gpu: {"job": "idle", "last": ""} for gpu in range(2)}
def pause_due():
    if time.time() >= VNA_DEADLINE - 600: session_paused.set()
    return session_paused.is_set()
def launch(cmd, gpu, name, log_path):
    from experiments.vna_resume import launch_logged
    return launch_logged(cmd, cwd=REPO, env=gpu_env(gpu), log_path=log_path,
                         lane=gpu, job=name, lanes=lane_state,
                         paused=session_paused, deadline=VNA_DEADLINE)
def heartbeat():
    from experiments.vna_resume import print_heartbeat
    while not archive_stop.wait(600):
        print_heartbeat(RUN_ROOT / "outputs", SESSION_START, VNA_DEADLINE, lane_state)
archive_lock = threading.Lock(); archive_stop = threading.Event(); archive_errors = []
def archive_outputs():
    with archive_lock:
        return _archive_outputs(RUN_ROOT, budget=DISK_CAP_BYTES,
                                folders=("outputs/vna_v5_inc10_p2",), exclude_parts=("data",))
def periodic_archive():
    while not archive_stop.wait(3600):
        try: archive_outputs()
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
def gpu_env(gpu): return {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
archive_errors = []
lane_timings = {}
archiver = threading.Thread(target=periodic_archive, daemon=True); archiver.start()
heartbeat_thread = threading.Thread(target=heartbeat, daemon=True); heartbeat_thread.start()


In [ ]:
import queue
from concurrent.futures import ThreadPoolExecutor
from experiments.io_utils import atomic_json_save
SCHEDULE=ROOT/'_schedules'/'inc10_p2_certswap.json'
errors=[]
report_stop=threading.Event()
report_lock=threading.Lock()

def read(path,default):
    try:return json.loads(path.read_text())
    except FileNotFoundError:return default

def identity(value,arm):
    cfg=value.get('cfg',{})
    if (value.get('code_commit')!=REPO_COMMIT or value.get('source_digest')!=SOURCE_DIGEST
            or value.get('arm')!=arm or value.get('seed')!=0
            or cfg.get('init_cls')!=10 or cfg.get('increment')!=10
            or value.get('warmup_epochs')!=1 or value.get('grad_clip') is not None
            or value.get('class_order_kind',value.get('order'))!='superclass'
            or value.get('backbone') not in ('nobn','resnet32_nobn')
            or not value.get('class_order_sha256')
            or value.get('fake_data') or value.get('preflight_only')):
        raise ValueError('P2 completed identity mismatch: '+arm)
    if arm=='icarl_full' and (value.get('policy_version'),value.get('solver'),value.get('curvature'))!=('certswap_t32_v1','gram','power10'):
        raise ValueError('P2 CertSwap solver/policy mismatch')

def job_name(arm):return arm+'__nobn__superclass__inc10__p2__s0'

def command(arm,out):
    common=['--increment','10','--warmup-epochs','1','--seed','0',
            '--nobn-lr-scale','.5','--device','cuda:0','--data-dir',str(DATA_ROOT),'--out',str(out)]
    if arm in ('icarl','icarl_full'):
        cmd=[sys.executable,'-u','-m','experiments.main_cil','--arm',arm,
             '--backbone','resnet32_nobn','--class-order','superclass','--solver','gram',
             '--curvature','power10','--max-tasks','10',*common]
        if arm=='icarl_full':cmd+=['--gir-rule','certswap','--certswap-topk','8','--certswap-lm-budget','20','--schedule',str(SCHEDULE)]
        return cmd
    return [sys.executable,'-u','-m','experiments.v4_competitors','--method',arm,
            '--backbone','nobn','--order','superclass','--resume',*common]

def prepare_schedule():
    if SCHEDULE.exists():
        value=read(SCHEDULE,{})
        expected=dict(authoritative=True,code_commit=REPO_COMMIT,source_digest=SOURCE_DIGEST,
            init_cls=10,increment=10,warmup_epochs=1,gir_rule='certswap',
            policy_version='certswap_t32_v1',solver='gram',curvature='power10')
        if any(value.get(k)!=v for k,v in expected.items()):raise ValueError('Inc10/P2 authority schedule mismatch')
        return True
    cmd=command('icarl_full',ROOT/'_preflight')
    position=cmd.index('--schedule');del cmd[position:position+2]
    cmd+=['--preflight-only','--schedule-output',str(SCHEDULE),'--schedule-source','self']
    if not launch(cmd,0,'inc10_p2_preflight',RUN_ROOT/'inc10_p2_preflight.log'):return False
    archive_outputs()
    if not SCHEDULE.exists():raise RuntimeError('Inc10/P2 preflight did not produce its own K0')
    value=read(SCHEDULE,{})
    print('INC10/P2 PREFLIGHT:',json.dumps(value,indent=2),flush=True)
    prepare_schedule()
    return True

def report():
    checkpoint2=[];final=[];activity=[];timings=[]
    for arm in ARMS:
        directory=ROOT/job_name(arm)
        summary=read(directory/'summary.json',{})
        tasks=read(directory/'tasks.json',summary.get('tasks',[]))
        events=read(directory/'interventions.json',[])
        diagnostics=read(directory/'diagnostics.json',[])
        if isinstance(diagnostics,dict):diagnostics=list(diagnostics.values())
        for diagnostic in diagnostics:
            activity.append(dict(arm=arm,task=diagnostic.get('task'),sites=diagnostic.get('sites',{})))
        for event in events:
            block=event.get('certswap',{})
            timings.append(dict(arm=arm,task=event.get('task'),iteration=event.get('iteration'),
                vna_seconds=event.get('vna_seconds'),certswap_seconds=event.get('certswap_seconds',block.get('seconds')),
                accepted=len(block.get('accepted',[])),rejections=block.get('rejections',{})))
        task2=next((row for row in tasks if row.get('task')==2),None)
        diverged=summary.get('diverged')
        task2_events=[r for r in timings if r['arm']==arm and r['task']==2]
        reasons={}
        for event in task2_events:
            for reason,count in event['rejections'].items():reasons[reason]=reasons.get(reason,0)+count
        if task2 or diverged:
            checkpoint2.append(dict(arm=arm,status=summary.get('status','RUNNING'),diverged=diverged,
                learning_accuracy=None if task2 is None else task2.get('cnn_learning_accuracy'),
                certswap_accepted=sum(r['accepted'] for r in task2_events),certswap_rejections=reasons))
        if len(tasks)==10 or summary.get('status') in ('DIVERGED','FAILED_DIVERGED'):
            last=tasks[-1] if tasks else {}
            final.append(dict(arm=arm,status=summary.get('status','RUNNING'),tasks=len(tasks),
                cnn_average=summary.get('cnn_average_accuracy',last.get('cnn_average_accuracy')),
                nme_average=summary.get('nme_average_accuracy',last.get('nme_average_accuracy')),
                cnn_last=summary.get('cnn_last_accuracy',last.get('cnn_top1')),
                nme_last=summary.get('nme_last_accuracy',last.get('nme_top1')),
                learning_accuracy=summary.get('cnn_learning_accuracy',last.get('cnn_learning_accuracy')),
                nme_learning_accuracy=summary.get('nme_learning_accuracy',last.get('nme_learning_accuracy')),
                forgetting=summary.get('cnn_forgetting',last.get('cnn_forgetting')),
                nme_forgetting=summary.get('nme_forgetting',last.get('nme_forgetting'))))
    swaps_by_task=[dict(task=task,accepted=sum(row['accepted'] for row in timings if row['task']==task)) for task in range(1,11)]
    result=dict(task2=checkpoint2,task10=final,activity_by_task=activity,applications=timings,swaps_by_task=swaps_by_task,errors=list(errors))
    with report_lock:atomic_json_save(result,ROOT/'P2_PILOT_REPORT.json')
    return result

def print_report(result):
    print('P2 TASK2:',json.dumps(result['task2'],indent=2),flush=True)
    print('P2 TASK10:',json.dumps(result['task10'],indent=2),flush=True)
    if result['task10']:
        print('P2 CERTSWAP BY TASK:',json.dumps(result['swaps_by_task'],indent=2),flush=True)
        print('P2 CERTSWAP BY APPLICATION:',json.dumps(result['applications'],indent=2),flush=True)
        print('P2 DEAD/DORMANT BY TASK:',json.dumps(result['activity_by_task'],indent=2),flush=True)

def watch():
    previous=None
    while not report_stop.wait(30):
        value=report();key=json.dumps((value['task2'],value['task10']),sort_keys=True)
        if key!=previous and (value['task2'] or value['task10']):print_report(value);previous=key

pending=queue.Queue()
for arm in ('icarl_cbp','icarl_redo'):pending.put(arm)

def lane(gpu,first):
    initial=iter(first)
    while not pause_due():
        try:arm=next(initial)
        except StopIteration:
            try:arm=pending.get_nowait()
            except queue.Empty:return
        directory=ROOT/job_name(arm)
        summary=read(directory/'summary.json',{})
        if summary:
            identity(summary.get('identity',{}),arm)
            if summary.get('status')=='COMPLETED':
                if len(summary.get('tasks',[]))!=10:raise ValueError('Completed P2 run must contain all ten tasks')
                continue
            if summary.get('status') in ('FAILED_DIVERGED','DIVERGED'):
                print('FAILED_DIVERGED',summary.get('diverged'),arm,flush=True);continue
        try:
            if arm=='icarl_full' and not prepare_schedule():return
            if not launch(command(arm,directory),gpu,job_name(arm),RUN_ROOT/(job_name(arm)+'.log')):return
            outcome=read(directory/'summary.json',{})
            if outcome.get('status') in ('FAILED_DIVERGED','DIVERGED'):
                print('FAILED_DIVERGED',outcome.get('diverged'),arm,flush=True)
        except Exception as error:
            outcome=read(directory/'summary.json',{})
            if outcome.get('status') in ('FAILED_DIVERGED','DIVERGED'):
                print('FAILED_DIVERGED',outcome.get('diverged'),arm,flush=True)
            else:
                errors.append(dict(arm=arm,error=f'{type(error).__name__}: {error}'))
                print('P2 JOB ERROR:',errors[-1],flush=True)
        archive_outputs()
        print_report(report())

reporter=threading.Thread(target=watch,daemon=True);reporter.start()
try:
    with ThreadPoolExecutor(max_workers=2) as pool:
        jobs=[pool.submit(lane,0,['icarl_full']),pool.submit(lane,1,['icarl'])]
        for job in jobs:job.result()
finally:
    report_stop.set();reporter.join(timeout=10)
    print_report(report())
    archive_stop.set();archiver.join(timeout=10);archive_outputs()
if session_paused.is_set():print('PAUSED: attach this pilot output and rerun to continue P2 only.')
if errors:raise RuntimeError('P2 jobs with errors: '+json.dumps(errors))
if archive_errors:raise RuntimeError(archive_errors)
